# Lab 05.1 — Spark e database relazionali (JDBC / MySQL)

Spark può leggere e scrivere qualunque database dotato di driver **JDBC**.
Usi tipici: importare tabelle in un data lake, unire dati del DB con file su HDFS/Parquet, scrivere risultati aggregati per applicazioni e report.

Il container `mysql` contiene il database `shop` con le tabelle `customers` (500 righe) e `orders` (20 000 righe), caricate da `initdb/01-shop.sql`.

Il driver JDBC viene scaricato da Maven al primo avvio con `spark.jars.packages` (conservato nel volume `ivy-cache`).

In [ ]:
import os, time
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("Lab05-MySQL")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.jars.packages", "com.mysql:mysql-connector-j:9.5.0")
         .config("spark.jars.ivy", "/home/jovyan/.ivy2")
         .getOrCreate())

URL = os.environ.get("MYSQL_URL", "jdbc:mysql://mysql:3306/shop")
PROPS = {"user": "spark", "password": "spark", "driver": "com.mysql.cj.jdbc.Driver"}

## 1. Leggere una tabella
Lo schema viene ricavato dai metadati del database (tipi SQL → tipi Spark).

In [ ]:
orders = spark.read.jdbc(URL, "orders", properties=PROPS)
customers = spark.read.jdbc(URL, "customers", properties=PROPS)

orders.printSchema()
orders.show(5)
print("Partizioni:", orders.rdd.getNumPartitions(), "(una sola connessione: lettura seriale)")

## 2. Pushdown verso il database
Filtri e proiezioni vengono tradotti in SQL ed eseguiti **da MySQL**: in Spark arrivano solo le righe necessarie.

In [ ]:
pagati = orders.where("status = 'PAID' AND amount > 200").select("order_id", "amount")
pagati.explain()      # PushedFilters: [IsNotNull(status), EqualTo(status,PAID), GreaterThan(amount,200)]
print("Righe:", pagati.count())

In [ ]:
# Si può anche delegare al DB un'intera query con l'opzione "query"
per_citta = (spark.read.format("jdbc")
             .option("url", URL).options(**PROPS)
             .option("query", """
                 SELECT c.city, COUNT(*) AS n_ordini, SUM(o.amount) AS totale
                 FROM orders o JOIN customers c ON o.customer_id = c.customer_id
                 GROUP BY c.city""")
             .load())
per_citta.orderBy(F.desc("totale")).show()

## 3. Lettura parallela
Indicando una colonna numerica e il suo intervallo, Spark divide la tabella in `numPartitions` range ed esegue **una query per partizione in parallelo**:
`WHERE order_id >= 1 AND order_id < 5001`, `WHERE order_id >= 5001 AND ...`, ...

In [ ]:
orders_par = spark.read.jdbc(URL, "orders", properties=PROPS,
                             column="order_id", lowerBound=1, upperBound=20001, numPartitions=4)
print("Partizioni:", orders_par.rdd.getNumPartitions())
print("Righe per partizione:", orders_par.rdd.glom().map(len).collect())

## 4. Analisi in Spark e unione con altri dati
Una volta caricati, i dati del DB sono normali DataFrame: si possono unire a dati provenienti da altre sorgenti (qui un piccolo DataFrame con gli obiettivi di vendita).

In [ ]:
obiettivi = spark.createDataFrame(
    [("Roma", 300_000), ("Milano", 450_000), ("Napoli", 250_000),
     ("Torino", 420_000), ("Bologna", 260_000), ("Firenze", 500_000)],
    "city STRING, obiettivo BIGINT")

risultati = (orders_par.where("status != 'CANCELLED'")
             .join(customers, "customer_id")
             .groupBy("city").agg(F.sum("amount").alias("venduto"))
             .join(obiettivi, "city")
             .withColumn("raggiunto_%", F.round(F.col("venduto") / F.col("obiettivo") * 100, 1))
             .orderBy(F.desc("raggiunto_%")))
risultati.show()

## 5. Scrivere su MySQL
`mode("overwrite")` ricrea la tabella; `append` aggiunge righe. Ogni partizione scrive in parallelo con la propria connessione (inserimenti a blocchi di `batchsize`).

In [ ]:
(risultati.write.mode("overwrite")
          .option("batchsize", 1000)
          .jdbc(URL, "risultati_citta", properties=PROPS))

spark.read.jdbc(URL, "risultati_citta", properties=PROPS).show()

In [ ]:
# Statistiche mensili scritte in una nuova tabella
mensili = (orders_par.groupBy(F.date_format("order_date", "yyyy-MM").alias("mese"), "status")
           .agg(F.count("*").alias("n"), F.round(F.sum("amount"), 2).alias("totale")))
mensili.write.mode("overwrite").jdbc(URL, "statistiche_mensili", properties=PROPS)
print(spark.read.jdbc(URL, "statistiche_mensili", properties=PROPS).count(), "righe scritte")

In [ ]:
spark.stop()